---
numbering: false
---

# 2.6: Affine lines and planes in ℝ³


In [1]:
import numpy as np
import plotly.graph_objects as go
from pathlib import Path
import sys

_notes_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'myst.yml').exists())
if str(_notes_root) not in sys.path:
    sys.path.insert(0, str(_notes_root))
from plot_style import style_plotly

BLUE, ORANGE, PINK = '#3d81f6', 'orange', '#d81a60'


def base3():
    fig=style_plotly(go.Figure(),renderer='plotly_mimetype')
    axis=dict(range=[-6,8],dtick=2,tickfont=dict(size=11),showbackground=True,showspikes=False,
              backgroundcolor='white',gridcolor='#e5e7eb',zerolinecolor='#9ca3af')
    fig.update_layout(autosize=True,height=520,showlegend=False,font=dict(size=16),
                      margin=dict(l=0,r=0,t=15,b=0),
                      scene=dict(bgcolor='white',xaxis=dict(title=dict(text='x',font=dict(size=13)),**axis),yaxis=dict(title=dict(text='y',font=dict(size=13)),**axis),
                                 zaxis=dict(title=dict(text='z',font=dict(size=13)),**axis),aspectmode='cube',
                                 camera=dict(eye=dict(x=1.6,y=-2.1,z=1.3))))
    for direction in np.eye(3):
        line3(fig,-6*direction,8*direction,'#9ca3af',width=2)
    return fig


def line3(fig,start,end,color=BLUE,width=5,dash='solid'):
    fig.add_trace(go.Scatter3d(x=[start[0],end[0]],y=[start[1],end[1]],z=[start[2],end[2]],
                              mode='lines',line=dict(color=color,width=width,dash=dash),
                              hoverinfo='skip',showlegend=False))


def vec3(fig,end,label,color=BLUE,start=(0,0,0),offset=(0.25,0.25,0.35)):
    start,end=np.asarray(start,float),np.asarray(end,float)
    direction=(end-start)/np.linalg.norm(end-start)
    line3(fig,start,end-0.15*direction,color,width=7)
    fig.add_trace(go.Cone(x=[end[0]],y=[end[1]],z=[end[2]],u=[direction[0]],v=[direction[1]],w=[direction[2]],
                         anchor='tip',sizemode='absolute',sizeref=0.45,colorscale=[[0,color],[1,color]],
                         showscale=False,hoverinfo='skip'))
    pos=end+offset
    fig.add_trace(go.Scatter3d(x=[pos[0]],y=[pos[1]],z=[pos[2]],mode='text',text=[label],
                              textfont=dict(family='Palatino',color=color,size=20),hoverinfo='skip'))


def plane3(fig,normal,color=BLUE,opacity=0.24,d=0,bounds=(-5,7)):
    # Solve n_x*x + n_y*y + n_z*z = d for y on a rectangular x,z mesh.
    x,z=np.meshgrid(np.linspace(*bounds,2),np.linspace(*bounds,2))
    n=np.asarray(normal,float)
    y=(d-n[0]*x-n[2]*z)/n[1]
    fig.add_trace(go.Surface(x=x.tolist(),y=y.tolist(),z=z.tolist(),
                            colorscale=[[0,color],[1,color]],opacity=opacity,
                            showscale=False,hoverinfo='skip'))
    corners=np.array([[x[0,0],y[0,0],z[0,0]],[x[0,1],y[0,1],z[0,1]],
                      [x[1,1],y[1,1],z[1,1]],[x[1,0],y[1,0],z[1,0]]])
    for i in range(4): line3(fig,corners[i],corners[(i+1)%4],color,width=2)




In [Chapter 2.4](02-04.ipynb), we described lines and planes through the origin in parametric form. In [Chapter 2.5](02-05.ipynb), we described them using linear equations of the form $$ax + by + cz = d,$$ where $d$ was forced to be $0$.

Let's now think about lines and planes in $\mathbb{R}^3$ that are **not** required to pass through the origin. These are called **affine** lines and planes.

As we discussed in [Chapter 2.1](02-01.ipynb) when we introduced affine lines in $\mathbb{R}^2$, the idea is to **add a fixed vector $\vec p$ to every vector in the set**. The direction(s) stay the same, but the starting point changes.

In [8]:
fig=base3()
plane3(fig,(1,-2,1),BLUE,0.18)
plane3(fig,(1,-2,1),ORANGE,0.18,d=3)
vec3(fig,(0,0,3),'<i>p</i>⃗',PINK,offset=(0.4,-0.5,0.3))
fig.add_trace(go.Scatter3d(x=[0],y=[0],z=[0],mode='markers',
                         marker=dict(color='black',size=4),hoverinfo='skip'))
fig.show()


---

## Translating a plane

Recall the plane

$$P=\operatorname{span}(\vec v_1,\vec v_2),\qquad
\vec v_1=\begin{bmatrix}3\\4\\5\end{bmatrix},\qquad
\vec v_2=\begin{bmatrix}5\\2\\-1\end{bmatrix}.$$

It has normal vector and equation

$$\vec w=\begin{bmatrix}1\\-2\\1\end{bmatrix},\qquad x-2y+z=0.$$

For example, take $\vec p=\begin{bmatrix}0\\0\\3\end{bmatrix}$ and translate our plane $P$ by $\vec p$. In vector-parametric form, this is the plane

$$\begin{bmatrix}x\\y\\z\end{bmatrix}
=\begin{bmatrix}0\\0\\3\end{bmatrix}
+a\begin{bmatrix}3\\4\\5\end{bmatrix}
+b\begin{bmatrix}5\\2\\-1\end{bmatrix},\qquad a,b\in\mathbb R.$$

Translation preserves the directions in the plane, so $\vec w=\begin{bmatrix}1\\-2\\1\end{bmatrix}$ is still a normal vector. A vector $\vec x=\begin{bmatrix}x\\y\\z\end{bmatrix}$ is on the translated plane exactly when $\vec x-\vec p$ lies in $P$. Therefore,

$$\begin{aligned}
\vec w\cdot(\vec x-\vec p)&=0,\\
\vec w\cdot\vec x&=\vec w\cdot\vec p,\\
x-2y+z&=3.
\end{aligned}$$

```{figure} #plot-26-affine
:label: fig-24-affine
:class: course-caption
:alt: The blue plane x minus 2y plus z equals zero is translated upward by p to the orange plane x minus 2y plus z equals three.

Adding $\vec p=\begin{bmatrix}0\\0\\3\end{bmatrix}$ translates $P$ to the parallel plane $x-2y+z=3$.
```

More generally, a plane has equation $ax+by+cz=d$, where the normal vector $\begin{bmatrix}a\\b\\c\end{bmatrix}$ is nonzero. If $d=0$, the equation is **homogeneous** and the plane passes through the origin. If $d\ne0$, it is **nonhomogeneous** and the plane does not pass through the origin.



---

## Translating a line

The same idea works for a line. Recall

$$\ell=\operatorname{span}(\vec v_1),\qquad
\vec v_1=\begin{bmatrix}3\\4\\5\end{bmatrix}.$$

In [Chapter 2.5](02-05.ipynb), we used the two independent normals

$$\vec w=\begin{bmatrix}1\\-2\\1\end{bmatrix},\qquad
\vec w'=\begin{bmatrix}4\\-3\\0\end{bmatrix}.$$

Both dot products with $\vec v_1$ are zero. The line is therefore the intersection of $x-2y+z=0$ and $4x-3y=0$. Adding $\vec p$ to each vector on $\ell=\operatorname{span}(\vec v_1)$ gives

$$\begin{bmatrix}x\\y\\z\end{bmatrix}
=\begin{bmatrix}0\\0\\3\end{bmatrix}
+t\begin{bmatrix}3\\4\\5\end{bmatrix},\qquad t\in\mathbb R.$$

Again, we haven't changed the direction of the line, so we can keep the same normal vectors $\vec w$ and $\vec w'$. Taking their dot products with $\vec p$ gives the new right-hand sides, $3$ and $0$:

$$\begin{cases}x-2y+z=3,\\4x-3y=0.\end{cases}$$

An affine line in $\mathbb R^3$ is the intersection of two planes with independent normal vectors. Translating changes the right-hand sides of their equations while preserving the direction of the line.



In [3]:

fig=base3()
plane3(fig,(1,-2,1),BLUE,0.22,d=3)
plane3(fig,(4,-3,0),ORANGE,0.22,d=0)
p=np.array([0,0,3]); v=np.array([3,4,5])
line3(fig,p-1.1*v,p+v,PINK,width=7)
vec3(fig,p+v,'<i>v</i>⃗<sub>1</sub>',PINK,start=p,offset=(0.6,0.3,0.5))
fig.add_trace(go.Scatter3d(x=[0],y=[0],z=[3],mode='markers+text',
    marker=dict(color='black',size=4),text=['<i>p</i>⃗'],textposition='top left',
    textfont=dict(family='Palatino',size=16),hoverinfo='skip'))
fig.show()




```{figure} #plot-26-affine-line
:label: fig-plot-26-affine-line
:class: course-caption
:alt: Two translated planes intersect along a pink affine line.

The two translated planes intersect in the affine line through $(0,0,3)$ with direction $\vec v_1$.
```





### Another example

Consider the affine line $L$ in scalar-parametric form:

$$x=1+2t,\qquad y=2-t,\qquad z=-1+3t,\qquad t\in\mathbb R.$$

Choose the normals $\vec n_1=\begin{bmatrix}0\\3\\1\end{bmatrix}$ and $\vec n_2=\begin{bmatrix}-3\\0\\2\end{bmatrix}$ which are perpendicular to this line's direction: their dot products with $\begin{bmatrix}2\\-1\\3\end{bmatrix}$ are $-3+3=0$ and $-6+6=0$. Their dot products with the starting point $\vec p=\begin{bmatrix}1\\2\\-1\end{bmatrix}$ are $5$ and $-5$. Thus,

$$L=\left\{\vec x\in\mathbb R^3:
\vec n_1\cdot\vec x=\vec n_1\cdot\vec p,\quad
\vec n_2\cdot\vec x=\vec n_2\cdot\vec p\right\},$$

or, in scalar equations,

$$\begin{cases}3y+z=5,\\-3x+2z=-5.\end{cases}$$

These planes intersect in $L$. To check, let $z=-1+3t$ in the system; the equations give $y=2-t$ and $x=1+2t$.

::::{tip} Activity 1
Consider

$$Q'=\begin{bmatrix}2\\-1\\4\end{bmatrix}
+\operatorname{span}\left(\begin{bmatrix}1\\2\\-1\end{bmatrix},
\begin{bmatrix}2\\-1\\3\end{bmatrix}\right).$$

1. Write vector-parametric and scalar-parametric forms, stating the possible parameter values.
2. Use the normal $\vec n=\begin{bmatrix}1\\-1\\-1\end{bmatrix}$ (check that it is perpendicular to both spanning vectors) to find the constant in $\vec n\cdot\vec x=\vec n\cdot\vec p$.
3. Write an equation $ax+by+cz=d$ for the plane. Does it contain the origin?

:::{tip} Solution
:class: dropdown

The vector-parametric form is

$$\begin{bmatrix}x\\y\\z\end{bmatrix}
=\begin{bmatrix}2\\-1\\4\end{bmatrix}
+s\begin{bmatrix}1\\2\\-1\end{bmatrix}
+t\begin{bmatrix}2\\-1\\3\end{bmatrix},\qquad s,t\in\mathbb R.$$

The scalar-parametric form is

$$\begin{aligned}x&=2+s+2t,\\y&=-1+2s-t,\\z&=4-s+3t,\end{aligned}\qquad s,t\in\mathbb R.$$

The constant is $\vec n\cdot\vec p=2+1-4=-1$. Hence $\vec n\cdot\vec x=-1$, or $x-y-z=-1$. The origin does not satisfy this equation.
:::
::::

An affine line or plane **need not** pass through the origin, but it can. Translation does not automatically make every right-hand side nonzero: each constant is determined by the corresponding dot product.





### Comparing representations in Desmos 3D

Open the [Desmos 3D Calculator](https://www.desmos.com/3d). We write position vectors as columns in the notes; Desmos takes their coordinates as an ordered triple.

- **Parametric line:** enter `(1+2t, 2-t, -1+3t)` for the affine line above. Widen the displayed bounds for $t$, for example to $-10\leq t\leq10$, to see more of the line. A finite interval displays only part of the infinite line.
- **Parametric plane:** Desmos uses $u,v$ as surface parameters. Rename our $s,t$ to $u,v$ and enter `(2+u+2v, -1+2u-v, 4-u+3v)`. Widen both parameter intervals to see a larger patch of the plane.
- **Plane equation:** enter `x-y-z=-1`. Compare this plane with the parametric patch; the patch should lie on it.
- **Line as an intersection:** enter `3y+z=5` and `-3x+2z=-5` separately. Each equation gives a plane. Their intersection should match the parametric line.

Rotate the view and toggle expressions to compare the graphs. Matching graphs illustrate the relationship; the dot-product calculations and substitutions above explain why the descriptions agree.

For more on the input syntax, see Desmos's [parametric equations guide](https://help.desmos.com/hc/en-us/articles/4406906208397-Parametric-Equations).

